# SPINE-GPE v7 — Layout Closure + RAIS Formal Baseline v1.0.0

Fluxo sequencial:

1. fechar ou congelar a pendência documental dos layouts PNADc;
2. auditar e certificar o baseline formal RAIS;
3. criar os locks para o futuro `Phase 0 Master Harmonization & Evidence Lock`.

Revise todas as células de configuração antes da execução.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, subprocess, sys, pandas as pd

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPTS = ROOT / 'scripts'
SCRIPTS.mkdir(parents=True, exist_ok=True)

PACKAGE_DIR = SCRIPTS  # copie os arquivos do pacote para esta pasta
REQ = PACKAGE_DIR / 'requirements_SPINE_GPEv7_LAYOUT_RAIS_PACKAGE_v1.0.0.txt'
LAYOUT_SCRIPT = PACKAGE_DIR / 'SPINE_GPEv7_PNADC_LAYOUT_EQUIVALENCE_CLOSURE_v1.0.0.py'
RAIS_SCRIPT = PACKAGE_DIR / 'SPINE_GPEv7_RAIS_FORMAL_CERTIFIER_v1.0.0.py'

for p in [REQ, LAYOUT_SCRIPT, RAIS_SCRIPT]:
    assert p.exists(), p
print('ROOT:', ROOT)


In [ ]:
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], text=True, capture_output=True)
print(install.stdout)
print(install.stderr)
assert install.returncode == 0


## 1. Configuração dos layouts PNADc

Adicione pastas que contenham dicionários, inputs ou arquivos de variáveis oficiais. O primeiro audit gera um registry template; copie-o, revise ano e independência documental, e informe o caminho em `LAYOUT_REGISTRY`.


In [ ]:
LAYOUT_YEARS = '2019,2020,2021,2022,2024'
LAYOUT_ROOTS = [
    ROOT / '01_raw' / 'IBGE',
]
LAYOUT_REGISTRY = None  # Ex.: ROOT/'00_admin/documentation/pnadc_layout_source_registry.csv'
HISTORICAL_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json'

LAYOUT_RUN_ID = 'layout_closure_v100'
cmd = [sys.executable, str(LAYOUT_SCRIPT), '--root', str(ROOT), '--mode', 'audit', '--run-id', LAYOUT_RUN_ID+'_audit', '--years', LAYOUT_YEARS, '--strict']
for r in LAYOUT_ROOTS:
    cmd += ['--layout-root', str(r)]
if HISTORICAL_LOCK.exists():
    cmd += ['--historical-lock', str(HISTORICAL_LOCK)]
audit = subprocess.run(cmd, text=True, capture_output=True)
print(audit.stdout)
print(audit.stderr)
print('exit:', audit.returncode)
assert audit.returncode == 0


In [ ]:
TABLE_LAYOUT = ROOT/'05_outputs/tables/pnadc_layout_equivalence_closure'
registry_templates = sorted(TABLE_LAYOUT.glob('pnadc_layout_source_registry_template_*.csv'))
print('Registry gerado:', registry_templates[-1] if registry_templates else 'não encontrado')
if registry_templates:
    display(pd.read_csv(registry_templates[-1]).head(50))


Revise o registry antes do full. Quando não houver documentos anuais independentes suficientes, o resultado esperado e cientificamente correto é `DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT`.


In [ ]:
cmd = [sys.executable, str(LAYOUT_SCRIPT), '--root', str(ROOT), '--mode', 'full', '--run-id', LAYOUT_RUN_ID, '--years', LAYOUT_YEARS, '--strict']
for r in LAYOUT_ROOTS:
    cmd += ['--layout-root', str(r)]
if LAYOUT_REGISTRY is not None:
    cmd += ['--source-registry', str(LAYOUT_REGISTRY)]
if HISTORICAL_LOCK.exists():
    cmd += ['--historical-lock', str(HISTORICAL_LOCK)]
full_layout = subprocess.run(cmd, text=True, capture_output=True)
print(full_layout.stdout)
print(full_layout.stderr)
print('exit:', full_layout.returncode)
assert full_layout.returncode == 0

layout_lock_path = ROOT/'00_admin/PNADC_LAYOUT_EQUIVALENCE_FINAL_LOCK.json'
layout_lock = json.loads(layout_lock_path.read_text(encoding='utf-8'))
print(json.dumps(layout_lock, ensure_ascii=False, indent=2))
assert layout_lock['status'] in ['LAYOUT_EQUIVALENCE_CONFIRMED','DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT']


## 2. Configuração RAIS

Aponte para as pastas/arquivos de vínculos. Não inclua arquivos de estabelecimentos. Para 2024, forneça o `De-Para Microdados.xlsx` oficial.

O universo principal é CBO `519110` e vínculo ativo em 31/12.


In [ ]:
RAIS_YEARS = '2017,2018,2019,2020,2021,2022,2023,2024'
RAIS_ROOTS = [
    ROOT / '01_raw' / 'MTE' / 'RAIS',
]
RAIS_SOURCE_FILES = []
DEPARA_2024 = ROOT / '00_admin' / 'documentation' / 'De-Para Microdados.xlsx'
DEFLATOR_CSV = None  # arquivo com year,factor_to_base; 2022=1
GOLDEN_CSV = None    # arquivo template do pacote, preenchido com valores oficiais
MINIMUM_WAGE_CSV = None
PRIMARY_CBO = '519110'
RAIS_RUN_ID = 'rais_formal_publication_v100'

cmd = [sys.executable, str(RAIS_SCRIPT), '--root', str(ROOT), '--mode', 'audit', '--run-id', RAIS_RUN_ID+'_audit', '--years', RAIS_YEARS, '--primary-cbo', PRIMARY_CBO]
for r in RAIS_ROOTS:
    cmd += ['--rais-root', str(r)]
for f in RAIS_SOURCE_FILES:
    cmd += ['--source-file', str(f)]
if DEPARA_2024.exists():
    cmd += ['--depara-2024', str(DEPARA_2024)]
rais_audit = subprocess.run(cmd, text=True, capture_output=True)
print(rais_audit.stdout)
print(rais_audit.stderr)
print('exit:', rais_audit.returncode)
assert rais_audit.returncode == 0


In [ ]:
audit_lock_path = ROOT/'00_admin/RAIS_FORMAL_AUDIT_LOCK.json'
rais_audit_lock = json.loads(audit_lock_path.read_text(encoding='utf-8'))
print(json.dumps(rais_audit_lock, ensure_ascii=False, indent=2))
source_audit_path = Path(rais_audit_lock['artifacts']['source_audit'])
display(pd.read_csv(source_audit_path).head(50))


## 3. Execução RAIS completa

O comando abaixo requer pelo menos seis anos certificados. Ajuste `--minimum-years` somente com justificativa documental.


In [ ]:
cmd = [
    sys.executable, str(RAIS_SCRIPT), '--root', str(ROOT), '--mode', 'full',
    '--run-id', RAIS_RUN_ID, '--years', RAIS_YEARS, '--primary-cbo', PRIMARY_CBO,
    '--chunksize', '300000', '--minimum-years', '6', '--real-base-year', '2022', '--strict'
]
for r in RAIS_ROOTS:
    cmd += ['--rais-root', str(r)]
for f in RAIS_SOURCE_FILES:
    cmd += ['--source-file', str(f)]
if DEPARA_2024.exists():
    cmd += ['--depara-2024', str(DEPARA_2024)]
if DEFLATOR_CSV is not None:
    cmd += ['--deflator-csv', str(DEFLATOR_CSV)]
if GOLDEN_CSV is not None:
    cmd += ['--golden-csv', str(GOLDEN_CSV)]
if MINIMUM_WAGE_CSV is not None:
    cmd += ['--minimum-wage-csv', str(MINIMUM_WAGE_CSV)]

rais_full = subprocess.run(cmd, text=True, capture_output=True)
print(rais_full.stdout)
print(rais_full.stderr)
print('exit:', rais_full.returncode)
assert rais_full.returncode == 0


In [ ]:
rais_lock_path = ROOT/'00_admin/RAIS_FORMAL_CERTIFICATION_LOCK.json'
rais_freeze_path = ROOT/'00_admin/RAIS_FORMAL_CORE_FREEZE.json'
rais_lock = json.loads(rais_lock_path.read_text(encoding='utf-8'))
rais_freeze = json.loads(rais_freeze_path.read_text(encoding='utf-8'))
print(json.dumps(rais_lock, ensure_ascii=False, indent=2))
print(json.dumps(rais_freeze, ensure_ascii=False, indent=2))
assert rais_lock['status'] == 'CORE_CERTIFIED'
assert rais_freeze['status'] == 'FROZEN'
assert rais_lock['evidence_tier'] == 'D'
assert rais_lock['platform_direct_observed'] is False


In [ ]:
TABLE_RAIS = ROOT/'05_outputs/tables/rais_formal_certification'
quality = pd.read_csv(rais_lock['artifacts']['quality'])
special = pd.read_csv(rais_lock['artifacts']['special_geographies'])
geography = pd.read_csv(rais_lock['artifacts']['geography'])
demographics = pd.read_csv(rais_lock['artifacts']['demographics'])
print('QUALIDADE')
display(quality)
print('BRASIL / NORDESTE / PE / RECIFE')
display(special)
print('DEMOGRAFIA')
display(demographics.head(100))


## 4. Gates finais

O bloco está pronto para entrar no futuro Master Lock apenas quando:

- o layout closure estiver confirmado ou congelado como limitação documental;
- a RAIS estiver `CORE_CERTIFIED` e `FROZEN`;
- o relatório mantiver a unidade como vínculo formal;
- plataforma direta e informalidade permanecerem explicitamente não observadas.


In [ ]:
assert layout_lock['status'] in ['LAYOUT_EQUIVALENCE_CONFIRMED','DOCUMENTATION_LIMITED_OPERATIONALLY_CONSISTENT']
assert rais_lock['status'] == 'CORE_CERTIFIED'
assert rais_freeze['read_only'] is True
print('LAYOUT CLOSURE + RAIS FORMAL READY FOR PHASE 0 MASTER HARMONIZATION LOCK')
